## Metrics Evaluation for Virtual Staining

# Note: there should be multiple real and virtual H&E image pairs in the respective directories

Three types of evaluations:
1. Pixel-based metrics: MSE, PSNR, SSIM
2. Perceptual metrics: FID, 
3. Nuclei-stats metrics: 

### Pixel-Based Metrics

1. MSE,RMSE, NRMSE
2. PSNR
3. SSIM, MS-SSIM

In [ ]:
import os

import cv2
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.metrics import (
    mean_squared_error,
    normalized_root_mse,
    peak_signal_noise_ratio,
    structural_similarity
)
from pytorch_msssim import ms_ssim
import torch

In [ ]:
def compute_metrics(real_HE_im, virtual_HE_im):
    # Assume uint8 or float [0,1]
    data_range = 255 if real_HE_im.dtype == np.uint8 else 1.0

    mse = mean_squared_error(real_HE_im, virtual_HE_im)
    rmse = np.sqrt(mse)
    nrmse = normalized_root_mse(real_HE_im, virtual_HE_im, normalization='euclidean')
    psnr = peak_signal_noise_ratio(real_HE_im, virtual_HE_im, data_range=data_range)
    min_dim = min(real_HE_im.shape[0], real_HE_im.shape[1])
    win_size = min(7, min_dim) if min_dim >= 3 else min_dim  # Ensure win_size is odd and <= min_dim
    if win_size % 2 == 0:
        win_size -= 1
    ssim = structural_similarity(
        real_HE_im, virtual_HE_im, data_range=data_range, channel_axis=-1, win_size=win_size
    )

    # MS-SSIM via pytorch, convert to tensor
    gt_t = torch.from_numpy(real_HE_im.transpose(2,0,1)[None,...]).float()
    fake_t = torch.from_numpy(virtual_HE_im.transpose(2,0,1)[None,...]).float()
    ms_ssim_val = ms_ssim(gt_t, fake_t, data_range=data_range, size_average=True).item()

    return mse, rmse, nrmse, psnr, ssim, ms_ssim_val

def pixel_wise_metrics_main(real_HE_dir, fake_HE_dir, output_dir):
    metrics = {k: [] for k in ['MSE','RMSE','NRMSE','PSNR','SSIM','MS-SSIM']} # Initialize metrics dictionary
    
    real_HE_list = [im_name for im_name in os.listdir(real_HE_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    fake_HE_list = [im_name for im_name in os.listdir(fake_HE_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    real_HE_list.sort()
    fake_HE_list.sort()
    assert len(real_HE_list) == len(fake_HE_list), "The number of real and virtual H&E images must be the same."
    print(f"Number of real H&E images: {len(real_HE_list)}, Number of virtual H&E images: {len(fake_HE_list)}")

    for i in range(len(real_HE_list)):
        real_HE_im = imread(os.path.join(real_HE_dir, real_HE_list[i]))
        virtual_HE_im = imread(os.path.join(fake_HE_dir, fake_HE_list[i]))
        # convert to float [0,1] for ss
        if real_HE_im.dtype == np.uint8:
            real_HE_im = real_HE_im.astype(np.float32) #/ 255.0
            virtual_HE_im = virtual_HE_im.astype(np.float32) #/ 255.0
        vals = compute_metrics(real_HE_im, virtual_HE_im)
        for k,v in zip(metrics.keys(), vals): # calculate every metric
            metrics[k].append(v)

    # Print mean and std for each metric, and store in a txt file
    metrics_txt_path = os.path.join(output_dir, "pixel_wise_metrics_results.txt")
    with open(metrics_txt_path, "w") as f:
        f.write("Metrics Summary:\n")
        for k in metrics:
            mean_val = np.mean(metrics[k])
            std_val = np.std(metrics[k])
            f.write(f"{k}: mean={mean_val:.4f}, std={std_val:.4f}\n")
            print(f"{k}: mean={mean_val:.4f}, std={std_val:.4f}")

    # Draw boxplots for each metric
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    axs = axs.flatten()
    for ax, k in zip(axs, metrics.keys()):
        ax.boxplot(metrics[k])
        ax.set_title(k)
        ax.set_ylabel(k)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "pixel_wise_metrics_boxplot.png"))
    plt.show()

In [ ]:
root_dir = "C:/Users/zpanp/projects/UTOM-master/datasets/test_data/250711_slides"
input_TPAF_dir = os.path.join(root_dir, "01_og_TPAF_RGB_patches")
real_HE_dir = os.path.join(root_dir, "01_og_real_HE_patches")
virtual_HE_dir = os.path.join(root_dir, "04_results_TPAF_gray_patches_nuc_replaced_with_vH")
metrics_results_dir = os.path.join(root_dir, "metrics_results_04_results_TPAF_gray_patches_nuc_replaced_with_vH")
if not os.path.exists(metrics_results_dir):
    os.makedirs(metrics_results_dir)

pixel_wise_metrics_main(real_HE_dir, virtual_HE_dir, metrics_results_dir)

## Perceptual Metrics

1. FID, KID
2. LPIPS
3. PCC (Pearson Correlation Coefficient)

In [ ]:
# Calculate perception metrics for evaluating virtual staining model: FID, KID, LPIPS, PCC
# Note: there should be multiple real and virtual H&E image pairs in the respective directories

import os
import tempfile
import shutil

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Fix for AttributeError: Can't pickle local object 'make_resizer.<locals>.func'
os.environ["CLEANFID_USE_GPU"] = "0"
os.environ["CLEANFID_USE_MULTIPROCESSING"] = "0"
from cleanfid import fid


In [ ]:
def compute_fid_kid(real_dir, fake_dir):
    fid_score = fid.compute_fid(real_dir, fake_dir, num_workers=0)
    kid_score = fid.compute_kid(real_dir, fake_dir, num_workers=0)
    return fid_score, kid_score

def compute_pcc_singlepair(real_img, fake_img):
    # flatten grayscale channels mean over RGB
    real_gray = real_img.mean(axis=2).ravel()
    fake_gray = fake_img.mean(axis=2).ravel()
    r, p = pearsonr(real_gray, fake_gray)
    return r

def perception_metrics_main(real_dir, fake_dir, output_dir):
    # 1. FID & KID via clean-fid (folder-level)
    fid_score, kid_score = compute_fid_kid(real_dir, fake_dir)
    print(f"FID: {fid_score:.4f}, KID: {kid_score:.4f}")

    # 2. PCC per-image
    real_HE_list = [im_name for im_name in os.listdir(real_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    fake_HE_list = [im_name for im_name in os.listdir(fake_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    real_HE_list.sort()
    fake_HE_list.sort()
    assert len(real_HE_list) == len(fake_HE_list), "The number of real and virtual H&E images must be the same."
    print(f"Number of real H&E images: {len(real_HE_list)}, Number of virtual H&E images: {len(fake_HE_list)}")
    pcc_list = []
    for i in range(len(real_HE_list)):
        real = cv2.imread(os.path.join(real_dir, real_HE_list[i]), cv2.IMREAD_UNCHANGED)
        fake = cv2.imread(os.path.join(fake_dir, fake_HE_list[i]), cv2.IMREAD_UNCHANGED)
        pcc = compute_pcc_singlepair(real, fake)
        pcc_list.append(pcc)
    print(f"PCC (mean ± std): {np.mean(pcc_list):.4f} ± {np.std(pcc_list):.4f}")

    # save FID, KID, and PCC numeric results in a text file
    with open(os.path.join(output_dir, 'perception_metrics_results.txt'), 'w') as f:
        f.write(f"FID: {fid_score:.4f}\n")
        f.write(f"KID: {kid_score:.4f}\n")
        f.write(f"PCC (mean ± std): {np.mean(pcc_list):.4f} ± {np.std(pcc_list):.4f}\n")

    # 3. Plotting
    fig, ax = plt.subplots(1, 3, figsize=(18,5))

    # Bar plot for FID & KID
    ax[0].bar(['FID','KID'], [fid_score, kid_score], color=['tab:blue','tab:orange'])
    ax[0].set_title('FID & KID between real & fake')
    ax[0].set_ylabel('Score')

    # Histogram + KDE for PCC distribution
    sns.histplot(pcc_list, bins=20, kde=True, ax=ax[1], color='purple')
    ax[1].set_title('Distribution of Pearson r per-image')
    ax[1].set_xlabel('Pearson r')

    # Scatter plot of PCC vs image index
    ax[2].scatter(range(len(pcc_list)), pcc_list, color='green')
    ax[2].axhline(np.mean(pcc_list), color='red', linestyle='--', label='Mean PCC')
    ax[2].set_title('PCC per image')
    ax[2].set_xlabel('Image index')
    ax[2].set_ylabel('Pearson r')
    ax[2].legend()

    plt.tight_layout()
    output_plot_path = os.path.join(output_dir, 'perception_metrics_plots.png')
    plt.savefig(output_plot_path)
    plt.show()


In [ ]:
root_dir = "C:/Users/zpanp/projects/UTOM-master/datasets/test_data/250711_slides"
input_TPAF_dir = os.path.join(root_dir, "01_og_TPAF_RGB_patches")
real_HE_dir = os.path.join(root_dir, "01_og_real_HE_patches")
virtual_HE_dir = os.path.join(root_dir, "03_results_vHE_TPAF_gray")
metrics_results_dir = os.path.join(root_dir, "metrics_results_03_results_vHE_TPAF_gray")
if not os.path.exists(metrics_results_dir):
    os.makedirs(metrics_results_dir)

perception_metrics_main(real_HE_dir, virtual_HE_dir, metrics_results_dir)

### Nuclei-Stats Metrics

1. 

In [ ]:
# Calculate nuclei statistics metrics for evaluating virtual staining model: nuclei count, area, and aspect ratio
# Note: there should be multiple real and virtual H&E image pairs in the respective directories

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.io import imread
from skimage.color import rgb2hed
from histomicstk.features import compute_nuclei_features
from skimage.measure import label

In [ ]:
def segment_nuclei_from_rgb(img_rgb):
    # 将 H&E 图转换为 hematoxylin 通道，简单阈值分割核
    hed = rgb2hed(img_rgb)
    h = hed[..., 0]
    # 使用 Otsu 等自定义阈值策略更优，此处以中位数为例
    thresh = np.median(h)
    mask = h < thresh
    label_mask = label(mask)
    return label_mask, h  # return segmented label mask and nuclei intensity

def get_nuclei_stats(im_label, im_nuclei):
    # compute morphometry features
    df = compute_nuclei_features(im_label, im_nuclei=im_nuclei,
                                 morphometry_features_flag=True,
                                 fsd_features_flag=False,
                                 intensity_features_flag=False,
                                 gradient_features_flag=False,
                                 haralick_features_flag=False)

    # select area and aspect ratio
    area = df['Size.Area']
    # aspect ratio: major_axis_length / minor_axis_length
    aspect = df['Size.MajorAxisLength'] / (df['Size.MinorAxisLength'] + 1e-6)
    aspect = np.clip(aspect, 0, 10)  # optionally clip to avoid extreme values
    return len(df), area.values, aspect.values

def nuclei_statistics_main(real_dir, fake_dir, output_dir):
    metrics = {'image': [], 'type': [], 'count': [], 'mean_area': [], 'mean_aspect': [], 'density': []}
    
    real_HE_list = [im_name for im_name in os.listdir(real_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    fake_HE_list = [im_name for im_name in os.listdir(fake_dir) if (im_name.endswith(".tif") or im_name.endswith(".tiff") or im_name.endswith(".png"))]
    real_HE_list.sort()
    fake_HE_list.sort()
    assert len(real_HE_list) == len(fake_HE_list), "The number of real and virtual H&E images must be the same."
    print(f"Number of real H&E images: {len(real_HE_list)}, Number of virtual H&E images: {len(fake_HE_list)}")

    for i in range(len(real_HE_list)):
        real = imread(os.path.join(real_dir, real_HE_list[i]))[..., :3]
        fake = imread(os.path.join(fake_dir, fake_HE_list[i]))[..., :3]
        for typ, img in [('real', real), ('fake', fake)]:
            lbl, nuc = segment_nuclei_from_rgb(img)
            cnt, areas, aspects = get_nuclei_stats(lbl, nuc)
            
            # nuclei density = nuclei count / image area (in pixels)
            img_area = img.shape[0] * img.shape[1]
            density = cnt / img_area

            metrics['image'].append(real_HE_list[i])
            metrics['type'].append(typ)
            metrics['count'].append(cnt)
            metrics['mean_area'].append(np.mean(areas) if len(areas) > 0 else 0)
            metrics['mean_aspect'].append(np.mean(aspects) if len(aspects) > 0 else 0)
            metrics['density'].append(density)

    df = pd.DataFrame(metrics)
    df.to_csv(os.path.join(output_dir, 'nuclei_statistics.csv'), index=False)
    print(df.groupby('type')[['count','mean_area','mean_aspect', 'density']].mean())

    # 绘图
    fig, axs = plt.subplots(1, 4, figsize=(24, 5))
    sns.boxplot(x='type', y='count', data=df, ax=axs[0])
    axs[0].set_title('Nuclei Count per Image')

    sns.boxplot(x='type', y='mean_area', data=df, ax=axs[1])
    axs[1].set_title('Mean Nucleus Area per Image')

    sns.boxplot(x='type', y='mean_aspect', data=df, ax=axs[2])
    axs[2].set_title('Mean Aspect Ratio per Image')

    sns.boxplot(x='type', y='density', data=df, ax=axs[3])
    axs[3].set_title('Nuclei Density per Image')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'nuclei_statistics_plots.png'))
    plt.show()



In [ ]:
root_dir = "C:/Users/zpanp/projects/UTOM-master/datasets/test_data/250711_slides"
input_TPAF_dir = os.path.join(root_dir, "01_og_TPAF_RGB_patches")
real_HE_dir = os.path.join(root_dir, "01_og_real_HE_patches")
virtual_HE_dir = os.path.join(root_dir, "04_results_TPAF_gray_patches_nuc_replaced_with_vH")
metrics_results_dir = os.path.join(root_dir, "metrics_results_04_results_TPAF_gray_patches_nuc_replaced_with_vH")
if not os.path.exists(metrics_results_dir):
    os.makedirs(metrics_results_dir)

nuclei_statistics_main(real_HE_dir, virtual_HE_dir, metrics_results_dir)